# Workspace Setup (Admin-only)

This notebook provisions shared resources for the course. Run it once per workspace before students begin.

**What it creates:**
1. Shared Unity Catalog namespace (`dbacademy.ka_all`) with volumes and PDFs
2. Vector Search endpoint (`vs_all_demo`) with user permissions
3. Two shared Vector Search indexes (Airbnb listings + house rules)
4. A shared Knowledge Assistant with the listings index attached
5. User permissions on all shared resources

In [0]:
%pip install "reportlab==4.4.10" "databricks-sdk==0.108.0" "databricks-vectorsearch==0.68" "pyyaml==6.0.3"
dbutils.library.restartPython()

In [0]:
import os
import shutil
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import iam
from databricks.vector_search.client import VectorSearchClient

from _lib.agent_bricks_manager import AgentBricksManager

ws = WorkspaceClient()
abm = AgentBricksManager(ws)
vsc = VectorSearchClient()

## 1. Define Variables and Create Catalog / Schema / Volume

Set up the shared UC namespace and define all variable names used throughout this notebook.

In [0]:
# --- Shared UC namespace ---
SHARED_CATALOG = "dbacademy"
SHARED_SCHEMA  = "ka_all"
SHARED_VOLUME  = "vol_all"
suffix = ""

# --- Knowledge Assistant ---
KA_NAME = f"Airbnb Knowledge Assistant{suffix}"

# --- Vector Search ---
VS_ENDPOINT_DEMO = f"vs_all_demo{suffix}"
EMBEDDING_MODEL  = "databricks-gte-large-en"

# Airbnb listings index
LISTINGS_CHUNKS_TABLE = f"{SHARED_CATALOG}.{SHARED_SCHEMA}.airbnb_listings_chunks{suffix}"
LISTINGS_VS_INDEX     = f"{SHARED_CATALOG}.{SHARED_SCHEMA}.airbnb_listings_index{suffix}"

# House rules index
HOUSE_RULES_CHUNKS_TABLE = f"{SHARED_CATALOG}.{SHARED_SCHEMA}.house_rules_chunks{suffix}"
HOUSE_RULES_VS_INDEX     = f"{SHARED_CATALOG}.{SHARED_SCHEMA}.house_rules_index{suffix}"

# --- Volume paths ---
VOLUME_PATH          = f"/Volumes/{SHARED_CATALOG}/{SHARED_SCHEMA}/{SHARED_VOLUME}"
LISTINGS_SUBFOLDER   = f"{VOLUME_PATH}/listings"
ADDITIONAL_SUBFOLDER = f"{VOLUME_PATH}/additional_listing_info"
INVOICES_SUBFOLDER   = f"{VOLUME_PATH}/invoices"
LISTINGS_PDF_PATH    = f"{LISTINGS_SUBFOLDER}/airbnb_listings.pdf"
HOUSE_RULES_PDF_PATH = f"{ADDITIONAL_SUBFOLDER}/airbnb_house_rules.pdf"

# --- External (Marketplace) dependencies ---
# Loaded outside this notebook from the Marketplace share; the grant in
# Section 6 assumes the table is already present.
DATABRICKS_SHARE_NAME = "dbacademy_airbnb"
TABLE_NAME            = "sf_airbnb_listings"

# Create catalog / schema / volume
spark.sql(f"USE CATALOG {SHARED_CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SHARED_SCHEMA}")
spark.sql(f"USE SCHEMA {SHARED_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SHARED_SCHEMA}.{SHARED_VOLUME}")

print(f"Shared catalog : {SHARED_CATALOG}")
print(f"Shared schema  : {SHARED_SCHEMA}")
print(f"Shared volume  : {VOLUME_PATH}")

## 2. Copy PDFs to Volume

Copy the source PDFs (listings, house rules, invoices) into the shared UC volume so they are accessible to all students.

In [0]:
# Source paths (relative to Includes/ when run as %run)
data_dir = os.path.join(os.getcwd(), "data")


def copy_file_to_volume(source_file, dest_path):
    """Copy a local file to a UC volume.

    Uses a direct Python copy instead of `dbutils.fs.cp` because on
    serverless compute the latter routes through Spark Connect, which
    can't read `file:/Workspace/...` source paths and raises a noisy
    `WorkspaceLocalFileSystemInternalError` even though the file is
    locally readable.
    """
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    shutil.copyfile(source_file, dest_path)
    print(f"  Copied: {os.path.basename(source_file)} -> {dest_path}")


# Create subfolders
os.makedirs(LISTINGS_SUBFOLDER, exist_ok=True)
os.makedirs(ADDITIONAL_SUBFOLDER, exist_ok=True)

# airbnb_listings.pdf -> vol_all/listings/
copy_file_to_volume(
    os.path.join(data_dir, "airbnb_listings.pdf"),
    f"{LISTINGS_SUBFOLDER}/airbnb_listings.pdf",
)

# airbnb_house_rules.pdf -> vol_all/additional_listing_info/
copy_file_to_volume(
    os.path.join(data_dir, "airbnb_house_rules.pdf"),
    f"{ADDITIONAL_SUBFOLDER}/airbnb_house_rules.pdf",
)

# invoices -> vol_all/invoices/
invoices_src_dir = os.path.join(data_dir, "invoices")
if os.path.exists(invoices_src_dir):
    os.makedirs(INVOICES_SUBFOLDER, exist_ok=True)
    for fname in sorted(os.listdir(invoices_src_dir)):
        if fname.endswith(".pdf"):
            copy_file_to_volume(
                os.path.join(invoices_src_dir, fname),
                f"{INVOICES_SUBFOLDER}/{fname}",
            )

print("\nAll PDFs copied to volume.")


## 3. Create Vector Search Endpoints

One endpoint is created:
- **`vs_all_demo`** — used by demo and lab notebooks and the shared indexes

In [0]:
def _ensure_vs_endpoint(endpoint_name):
    """Create a VS endpoint if it doesn't exist and wait for ONLINE."""
    try:
        vsc.get_endpoint(endpoint_name)
        print(f"Endpoint '{endpoint_name}' already exists.")
    except Exception:
        vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
        print(f"Creating endpoint '{endpoint_name}'...")

    for _ in range(60):
        ep = vsc.get_endpoint(endpoint_name)
        status = ep.get("endpoint_status", {}).get("state", "")
        if status == "ONLINE":
            print(f"Endpoint '{endpoint_name}' is ONLINE.")
            return
        print(f"  {endpoint_name} status: {status} - waiting...")
        time.sleep(10)
    print(f"WARNING: Endpoint '{endpoint_name}' did not reach ONLINE status in time.")

endpoint_list = [VS_ENDPOINT_DEMO]

for ep_name in endpoint_list:
    _ensure_vs_endpoint(ep_name)

### 3a. Grant `CAN_USE` on Endpoints

VS endpoints are workspace objects, so permissions are managed via the Permissions API. Without `CAN_USE`, users cannot query indexes hosted on these endpoints.

In [0]:
for ep_name in endpoint_list:
    ep = vsc.get_endpoint(ep_name)
    ep_id = str(ep["id"])
    ws.permissions.set(
        request_object_type="vector-search-endpoints",
        request_object_id=ep_id,
        access_control_list=[
            iam.AccessControlRequest(
                group_name="users",
                permission_level=iam.PermissionLevel.CAN_USE,
            )
        ],
    )
    print(f"Granted CAN_USE on endpoint '{ep_name}' (id={ep_id}) to users.")

## 4. Build Shared Vector Search Indexes

### 4a. Airbnb Listings Index

Parse `airbnb_listings.pdf`, chunk by property blocks, and create a delta-sync VS index.

In [0]:
# Parse and chunk the PDF using ai_parse_document
parsed_df = spark.sql(f"""
  WITH parsed_docs AS (
    SELECT
      path,
      ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES(
      '{LISTINGS_PDF_PATH}',
      format => 'binaryFile'
    )
  )
  SELECT
    path,
    element.value:element_type::STRING AS element_type,
    element.value:content::STRING      AS content
  FROM
    parsed_docs,
    LATERAL variant_explode(parsed_docs.parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
""")

display(parsed_df.limit(5))

In [0]:
import uuid as _uuid

CHUNK_SIZE    = 800
CHUNK_OVERLAP = 200


def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


# Group elements into per-listing blocks (delimited by "Property ID:" markers)
rows = parsed_df.collect()
listings = []
current_block = []
for row in rows:
    content = row["content"] or ""
    if content.startswith("Property ID:") and current_block:
        listings.append("\n".join(current_block))
        current_block = []
    current_block.append(content)
if current_block:
    listings.append("\n".join(current_block))

# Chunk each listing block
chunk_records = []
for listing_text in listings:
    for chunk in chunk_text(listing_text):
        chunk_records.append({
            "chunk_id":    str(_uuid.uuid4()),
            "chunk_text":  chunk.strip(),
            "source_path": LISTINGS_PDF_PATH,
        })

print(f"Created {len(chunk_records)} chunks from {len(listings)} listing blocks")

# Write chunks to a CDF-enabled Delta table
chunks_df = spark.createDataFrame(chunk_records)
(
    chunks_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(LISTINGS_CHUNKS_TABLE)
)

spark.sql(
    f"ALTER TABLE {LISTINGS_CHUNKS_TABLE} "
    f"SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)
spark.sql(f"GRANT SELECT ON TABLE {LISTINGS_CHUNKS_TABLE} TO `account users`")

print(f"Chunks table written: {LISTINGS_CHUNKS_TABLE}")

In [0]:
# Create the vector search index
try:
    index = vsc.get_index(endpoint_name=VS_ENDPOINT_DEMO, index_name=LISTINGS_VS_INDEX)
    print(f"Index '{LISTINGS_VS_INDEX}' already exists.")
except Exception:
    index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT_DEMO,
        source_table_name=LISTINGS_CHUNKS_TABLE,
        index_name=LISTINGS_VS_INDEX,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=EMBEDDING_MODEL,
        columns_to_sync=["chunk_id", "chunk_text", "source_path"],
    )
    print(f"Index '{LISTINGS_VS_INDEX}' created. Syncing...")

print(index.describe())

Grant `SELECT` on the listings index so users can query it.

In [0]:
spark.sql(f"GRANT SELECT ON TABLE {LISTINGS_VS_INDEX} TO `account users`")
print(f"Granted SELECT on {LISTINGS_VS_INDEX} to users.")

### 4b. House Rules Index

Parse `airbnb_house_rules.pdf` using `ai_prep_search` for semantic chunking, and create a delta-sync VS index.

In [0]:
# Parse and chunk the house rules PDF using ai_prep_search
house_rules_chunks_df = spark.sql(f"""
  WITH parsed_documents AS (
    SELECT
      path,
      ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{HOUSE_RULES_PDF_PATH}', format => 'binaryFile')
  ),
  prepped_documents AS (
    SELECT
      path,
      ai_prep_search(parsed) AS result
    FROM parsed_documents
  )
  SELECT
    uuid() AS chunk_id,
    chunk.value:chunk_to_embed::STRING AS chunk_text,
    chunk.value:chunk_to_retrieve::STRING AS retrieval_text,
    p.path AS source_path
  FROM
    prepped_documents p,
    LATERAL variant_explode(p.result:document.contents) AS chunk
  WHERE chunk.value:chunk_to_embed IS NOT NULL
""")

(
    house_rules_chunks_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(HOUSE_RULES_CHUNKS_TABLE)
)

spark.sql(
    f"ALTER TABLE {HOUSE_RULES_CHUNKS_TABLE} "
    f"SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)
spark.sql(f"GRANT SELECT ON TABLE {HOUSE_RULES_CHUNKS_TABLE} TO `account users`")

print(f"House rules chunks table written: {HOUSE_RULES_CHUNKS_TABLE}")

In [0]:
# Create the house rules vector search index
try:
    hr_index = vsc.get_index(endpoint_name=VS_ENDPOINT_DEMO, index_name=HOUSE_RULES_VS_INDEX)
    print(f"Index '{HOUSE_RULES_VS_INDEX}' already exists.")
except Exception:
    hr_index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT_DEMO,
        source_table_name=HOUSE_RULES_CHUNKS_TABLE,
        index_name=HOUSE_RULES_VS_INDEX,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=EMBEDDING_MODEL,
        columns_to_sync=["chunk_id", "chunk_text", "source_path"],
    )
    print(f"Index '{HOUSE_RULES_VS_INDEX}' created. Syncing...")

print(hr_index.describe())

In [0]:
spark.sql(f"GRANT SELECT ON TABLE {HOUSE_RULES_VS_INDEX} TO `account users`")
print(f"Granted SELECT on {HOUSE_RULES_VS_INDEX} to users.")

## 5. Create the Shared Knowledge Assistant

Create the **Airbnb Knowledge Assistant** and attach the listings VS index as a knowledge source.

In [0]:
existing_ka = abm.find_by_name(KA_NAME)

if existing_ka:
    ka_tile_id = existing_ka.tile_id
    print(f"Knowledge Assistant already exists: {KA_NAME} (tile_id: {ka_tile_id})")
    ka      = abm.ka_get(ka_tile_id)
    ka_name = ka["knowledge_assistant"]["name"]   # full resource path: "knowledge-assistants/<id>"
else:
    print(f"Creating Knowledge Assistant: {KA_NAME}...")
    # The REST endpoint /api/2.0/knowledge-assistants requires a non-empty
    # `knowledge_sources` list at create time. The SDK's
    # create_knowledge_assistant does not, so we create the KA via SDK and
    # attach the VS index as a separate source below.
    from databricks.sdk.service.knowledgeassistants import KnowledgeAssistant

    created = ws.knowledge_assistants.create_knowledge_assistant(
        knowledge_assistant=KnowledgeAssistant(
            display_name=KA_NAME,
            description=(
                "Knowledge Assistant for Airbnb listings data. Ask questions "
                "about property details, pricing, and neighborhoods."
            ),
            instructions=(
                "You are a helpful assistant that answers questions about Airbnb "
                "listings in San Francisco. Use the provided documents to find "
                "relevant information about properties, pricing, and neighborhoods."
            ),
        )
    )
    ka_name = created.name   # "knowledge-assistants/<id>"

    # The manager's wait/poll helpers key off tile_id; resolve it via find_by_name.
    found = abm.find_by_name(KA_NAME)
    if not found:
        raise RuntimeError(
            f"Created KA '{KA_NAME}' via SDK but could not locate its tile_id."
        )
    ka_tile_id = found.tile_id
    print(f"  Created KA: name={ka_name}, tile_id={ka_tile_id}")

# Resolve ka_name via the SDK to ensure consistent format for list_knowledge_sources
ka_sdk = next(ka for ka in ws.knowledge_assistants.list_knowledge_assistants() if ka.display_name == KA_NAME)
ka_name = ka_sdk.name

# Attach the VS index as a knowledge source (skip if already attached)
existing_sources = list(ws.knowledge_assistants.list_knowledge_sources(parent=ka_name))
already_attached = any(
    s.index and s.index.index_name == LISTINGS_VS_INDEX
    for s in existing_sources
)

if already_attached:
    print(f"Knowledge source '{LISTINGS_VS_INDEX}' already attached — skipping.")
else:
    source = abm.ka_add_index_source(
        parent_name=ka_name,
        index_name=LISTINGS_VS_INDEX,
        text_col="chunk_text",
        doc_uri_col="source_path",
        display_name="Airbnb Listings Vector Index",
        description="Vector search index over parsed Airbnb listing descriptions",
    )
    print(f"Knowledge Source added: {source.name}")

# Wait for the KA endpoint to come online
print("Waiting for KA endpoint to come online (this may take several minutes)...")
try:
    abm.ka_wait_until_endpoint_online(ka_tile_id, timeout_s=600)
    print("KA endpoint is ONLINE and ready for students.")
except TimeoutError as e:
    print(f"Warning: {e}")
    print("The KA may still be provisioning. Students can proceed once it's online.")

## 6. Grant Permissions

Grant access to the KA, its serving endpoint, and all shared catalog resources.

In [0]:
import time
import requests as _requests

_headers = ws.config.authenticate()
_headers["Content-Type"] = "application/json"

# 6a. Share KA with all users (CAN_MANAGE so they can explore configuration)
resp = _requests.patch(
    f"{ws.config.host}/api/2.0/permissions/knowledge-assistants/{ka_tile_id}",
    headers=_headers,
    json={
        "access_control_list": [
            {"group_name": "users", "permission_level": "CAN_MANAGE"}
        ]
    },
    timeout=30,
)
resp.raise_for_status()
print(f"Shared KA '{KA_NAME}' with all users (CAN_MANAGE)")

# 6b. Grant CAN_QUERY on the KA's serving endpoint.
# The Permissions API requires the endpoint ID (not the name) in the URL path.
kas = ws.knowledge_assistants.list_knowledge_assistants()
ka_details = next(ka for ka in kas if ka.display_name == KA_NAME)
ka_endpoint_name = ka_details.endpoint_name
print(f"KA serving endpoint: {ka_endpoint_name}")

# Retrieve the serving endpoint ID from the endpoint details
ep_resp = _requests.get(
    f"{ws.config.host}/api/2.0/serving-endpoints/{ka_endpoint_name}",
    headers=_headers,
    timeout=30,
)
ep_resp.raise_for_status()
ka_endpoint_id = ep_resp.json()["id"]

resp = _requests.patch(
    f"{ws.config.host}/api/2.0/permissions/serving-endpoints/{ka_endpoint_id}",
    headers=_headers,
    json={
        "access_control_list": [
            {"group_name": "users", "permission_level": "CAN_QUERY"}
        ]
    },
    timeout=30,
)
resp.raise_for_status()
print(f"Granted CAN_QUERY on serving endpoint '{ka_endpoint_name}' (id={ka_endpoint_id}) to all users")

In [0]:
# 6c. Grant CAN_READ on the MLflow experiment created by the Knowledge Assistant.
# The KA exposes its experiment_id directly — use that instead of guessing the name.
import mlflow

# Retry up to 3 times with a 20-second delay (experiment may not exist yet while KA provisions)
MAX_RETRIES = 3
RETRY_DELAY_S = 20
experiment = None

for attempt in range(1, MAX_RETRIES + 1):
    # Use the experiment_id from the KA SDK object (most reliable)
    if ka_sdk.experiment_id:
        try:
            experiment = mlflow.get_experiment(str(ka_sdk.experiment_id))
        except Exception:
            experiment = None
    if experiment:
        break
    if attempt < MAX_RETRIES:
        print(f"  Attempt {attempt}/{MAX_RETRIES}: Experiment not found yet. Retrying in {RETRY_DELAY_S}s...")
        time.sleep(RETRY_DELAY_S)
        # Refresh KA details in case experiment_id was just assigned
        ka_sdk = next(ka for ka in ws.knowledge_assistants.list_knowledge_assistants() if ka.display_name == KA_NAME)

if experiment:
    experiment_id = experiment.experiment_id
    resp = _requests.patch(
        f"{ws.config.host}/api/2.0/permissions/experiments/{experiment_id}",
        headers=_headers,
        json={
            "access_control_list": [
                {"group_name": "users", "permission_level": "CAN_READ"}
            ]
        },
        timeout=30,
    )
    resp.raise_for_status()
    print(f"Granted CAN_READ on MLflow experiment '{experiment.name}' (id={experiment_id}) to all users")
else:
    print(
        f"Warning: Could not find MLflow experiment after {MAX_RETRIES} attempts.\n"
        f"  The KA endpoint may still be provisioning. Re-run this cell once the KA is fully online."
    )

In [0]:
# Read-only access on shared catalog resources
spark.sql(f"GRANT USE CATALOG ON CATALOG {SHARED_CATALOG} TO `account users`")
spark.sql(f"GRANT USE SCHEMA  ON SCHEMA  {SHARED_CATALOG}.{SHARED_SCHEMA} TO `account users`")
spark.sql(f"GRANT READ VOLUME ON VOLUME  {SHARED_CATALOG}.{SHARED_SCHEMA}.{SHARED_VOLUME} TO `account users`")

# Marketplace-loaded Airbnb table — grant only if it exists in this workspace
try:
    spark.sql(
        f"GRANT SELECT ON TABLE "
        f"{SHARED_CATALOG}.{SHARED_SCHEMA}.{TABLE_NAME} TO `account users`"
    )
    print(f"Granted SELECT on {SHARED_CATALOG}.{SHARED_SCHEMA}.{TABLE_NAME}")
except Exception as e:
    print(
        f"Skipped GRANT on {TABLE_NAME}: {e}\n"
        f"  Load the table from the '{DATABRICKS_SHARE_NAME}' Marketplace "
        f"share, then re-run this cell."
    )

print(f"Granted read-only access on {SHARED_CATALOG}.{SHARED_SCHEMA} to all users")

## Summary

In [0]:
print("=" * 60)
print("Workspace Setup Complete")
print("=" * 60)
print(f"  Shared catalog   : {SHARED_CATALOG}")
print(f"  Shared schema    : {SHARED_SCHEMA}")
print(f"  Shared volume    : {VOLUME_PATH}")
print(f"  Listings PDF     : {LISTINGS_PDF_PATH}")
print(f"  VS endpoint demo : {VS_ENDPOINT_DEMO}")
print(f"  VS index (listings)    : {LISTINGS_VS_INDEX}")
print(f"  VS index (house rules) : {HOUSE_RULES_VS_INDEX}")
print(f"  KA name          : {KA_NAME}")
print(f"  KA tile_id       : {ka_tile_id}")
print(f"  User permissions : READ-ONLY on shared resources")